In [1]:
%pip install fg-data-profiling  --quiet
import pandas as pd
import numpy as np
import data_profiling 
from data_profiling import ProfileReport  # importing all requierd modules
import re


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
df = pd.read_json(r'D:\PROJECTS\London Food Retail Insights\0_Data\2_fsa_london.json') # reaing the data

pd.set_option('display.max.columns', 30)  # setting Columns to 30 so every column appears    

df.head()


,AddressLine1,AddressLine2,AddressLine3,AddressLine4,BusinessName,BusinessType,BusinessTypeID,ChangesByServerID,Distance,FHRSID,LocalAuthorityBusinessID,LocalAuthorityCode,LocalAuthorityEmailAddress,LocalAuthorityName,LocalAuthorityWebSite,NewRatingPending,Phone,PostCode,RatingDate,RatingKey,RatingValue,RightToReply,SchemeType,geocode,scores,borough
0,,309 Wood Lane,,Dagenham,5 Elms Cafe,Restaurant/Cafe/Canteen,1,0,NaN,1714830,81398,501,foodsafety@lbbd.gov.uk,Barking and Dagenham,http://www.lbbd.gov.uk/Pages/Home.aspx,False,,RM8 3NH,2024-07-13T00:00:00,fhrs_5_en-gb,5,,FHRS,"{'longitude': '0.142421', 'latitude': '51.5544...","{'Hygiene': 5, 'Structural': 5, 'ConfidenceInM...",Barking and Dagenham
1,,446 Becontree Avenue,,Dagenham,7 TILL 11,Retailers - other,4613,0,NaN,115956,45140,501,foodsafety@lbbd.gov.uk,Barking and Dagenham,http://www.lbbd.gov.uk/Pages/Home.aspx,False,,RM8 3UB,2023-09-15T00:00:00,fhrs_5_en-gb,5,,FHRS,"{'longitude': '0.129392', 'latitude': '51.5584...","{'Hygiene': 5, 'Structural': 5, 'ConfidenceInM...",Barking and Dagenham
2,,460 Lodge Avenue,,Dagenham,Aafio Mini Market,Retailers - other,4613,0,NaN,1413686,77594,501,foodsafety@lbbd.gov.uk,Barking and Dagenham,http://www.lbbd.gov.uk/Pages/Home.aspx,False,,RM9 4QS,2023-09-02T00:00:00,fhrs_5_en-gb,5,,FHRS,"{'longitude': '0.1106463', 'latitude': '51.534...","{'Hygiene': 5, 'Structural': 5, 'ConfidenceInM...",Barking and Dagenham
3,,North Street,,Barking,Abbey Children's Centre Day Nursery,Caring Premises,5,0,NaN,128460,58427,501,foodsafety@lbbd.gov.uk,Barking and Dagenham,http://www.lbbd.gov.uk/Pages/Home.aspx,False,,IG11 8JA,2026-05-27T00:00:00,fhrs_5_en-gb,5,,FHRS,"{'longitude': '0.07525', 'latitude': '51.541375'}","{'Hygiene': 5, 'Structural': 5, 'ConfidenceInM...",Barking and Dagenham
4,,1 Hewett Road,,Dagenham,Abbey Kebab & Pizza,Takeaway/sandwich shop,7844,0,NaN,122900,5136,501,foodsafety@lbbd.gov.uk,Barking and Dagenham,http://www.lbbd.gov.uk/Pages/Home.aspx,False,,RM8 2XT,2024-07-13T00:00:00,fhrs_5_en-gb,5,,FHRS,"{'longitude': '0.128666', 'latitude': '51.55093'}","{'Hygiene': 5, 'Structural': 5, 'ConfidenceInM...",Barking and Dagenham


In [3]:
# Normilzing the dictionries in a list 
df = pd.json_normalize(df.to_dict(orient="records")) 


df.head()  # verifying the fix applied

,AddressLine1,AddressLine2,AddressLine3,AddressLine4,BusinessName,BusinessType,BusinessTypeID,ChangesByServerID,Distance,FHRSID,LocalAuthorityBusinessID,LocalAuthorityCode,LocalAuthorityEmailAddress,LocalAuthorityName,LocalAuthorityWebSite,NewRatingPending,Phone,PostCode,RatingDate,RatingKey,RatingValue,RightToReply,SchemeType,borough,geocode.longitude,geocode.latitude,scores.Hygiene,scores.Structural,scores.ConfidenceInManagement
0,,309 Wood Lane,,Dagenham,5 Elms Cafe,Restaurant/Cafe/Canteen,1,0,NaN,1714830,81398,501,foodsafety@lbbd.gov.uk,Barking and Dagenham,http://www.lbbd.gov.uk/Pages/Home.aspx,False,,RM8 3NH,2024-07-13T00:00:00,fhrs_5_en-gb,5,,FHRS,Barking and Dagenham,0.142421,51.554493,5.0,5.0,5.0
1,,446 Becontree Avenue,,Dagenham,7 TILL 11,Retailers - other,4613,0,NaN,115956,45140,501,foodsafety@lbbd.gov.uk,Barking and Dagenham,http://www.lbbd.gov.uk/Pages/Home.aspx,False,,RM8 3UB,2023-09-15T00:00:00,fhrs_5_en-gb,5,,FHRS,Barking and Dagenham,0.129392,51.558435,5.0,5.0,5.0
2,,460 Lodge Avenue,,Dagenham,Aafio Mini Market,Retailers - other,4613,0,NaN,1413686,77594,501,foodsafety@lbbd.gov.uk,Barking and Dagenham,http://www.lbbd.gov.uk/Pages/Home.aspx,False,,RM9 4QS,2023-09-02T00:00:00,fhrs_5_en-gb,5,,FHRS,Barking and Dagenham,0.1106463,51.5347185,5.0,5.0,0.0
3,,North Street,,Barking,Abbey Children's Centre Day Nursery,Caring Premises,5,0,NaN,128460,58427,501,foodsafety@lbbd.gov.uk,Barking and Dagenham,http://www.lbbd.gov.uk/Pages/Home.aspx,False,,IG11 8JA,2026-05-27T00:00:00,fhrs_5_en-gb,5,,FHRS,Barking and Dagenham,0.07525,51.541375,5.0,5.0,5.0
4,,1 Hewett Road,,Dagenham,Abbey Kebab & Pizza,Takeaway/sandwich shop,7844,0,NaN,122900,5136,501,foodsafety@lbbd.gov.uk,Barking and Dagenham,http://www.lbbd.gov.uk/Pages/Home.aspx,False,,RM8 2XT,2024-07-13T00:00:00,fhrs_5_en-gb,5,,FHRS,Barking and Dagenham,0.128666,51.55093,5.0,5.0,5.0


In [8]:
# Using the ProfileReport Module to create data in a HTML file for Explortion 

#profile = ProfileReport(df , title = 'Profile Report', explorative = True)
#profile.to_file('report.html')
#file.download('report.html')

In [4]:
# ==========================================
# Data Explortion
# ==========================================

print(df.dtypes)    # need to look further in for columns RatingDate, RatingValue

print('Looking Missing Values')

print(df.isnull().sum())  # distance,geocode.longitude, geocode.latitude, scores.Hygiene, scores.Structural, scores.ConfidenceInManagement                
                   # All have missing values specialy Disatance has all values missing 
               
    

df.sample(5)    # Taking sample of a random 5 rows 

AddressLine1                      object
AddressLine2                      object
AddressLine3                      object
AddressLine4                      object
BusinessName                      object
BusinessType                      object
BusinessTypeID                     int64
ChangesByServerID                  int64
Distance                         float64
FHRSID                             int64
LocalAuthorityBusinessID          object
LocalAuthorityCode                 int64
LocalAuthorityEmailAddress        object
LocalAuthorityName                object
LocalAuthorityWebSite             object
NewRatingPending                    bool
Phone                             object
PostCode                          object
RatingDate                        object
RatingKey                         object
RatingValue                       object
RightToReply                      object
SchemeType                        object
borough                           object
geocode.longitud

,AddressLine1,AddressLine2,AddressLine3,AddressLine4,BusinessName,BusinessType,BusinessTypeID,ChangesByServerID,Distance,FHRSID,LocalAuthorityBusinessID,LocalAuthorityCode,LocalAuthorityEmailAddress,LocalAuthorityName,LocalAuthorityWebSite,NewRatingPending,Phone,PostCode,RatingDate,RatingKey,RatingValue,RightToReply,SchemeType,borough,geocode.longitude,geocode.latitude,scores.Hygiene,scores.Structural,scores.ConfidenceInManagement
48498,22 Lewin Road,London,,,Baptist Church,Other catering premises,7841,0,NaN,483299,06/01019/COMM,522,foodhealthandsafety@lambeth.gov.uk,Lambeth,http://www.lambeth.gov.uk,False,,SW16 6JR,2009-06-19T00:00:00,fhrs_5_en-gb,5,,FHRS,Lambeth,-0.132462,51.422809,0.0,0.0,0.0
44381,Tiffin Tree Express,61-63 Charterhouse Street,London,EC1M 6HA,Tiffin Tree Express,Restaurant/Cafe/Canteen,1,0,NaN,1890536,PI/000297973,519,Commercial.Envh@islington.gov.uk,Islington,http://www.islington.gov.uk,False,,EC1M 6HA,2026-01-20T00:00:00,fhrs_4_en-gb,4,,FHRS,Islington,-0.10349475,51.51916383,5.0,5.0,10.0
45421,25G Lowndes Street,LONDON,,,EL&N Cafe,Restaurant/Cafe/Canteen,1,0,NaN,1126983,INSPF/19/236993,520,food.hygiene@rbkc.gov.uk,Kensington and Chelsea,http://www.rbkc.gov.uk,False,,SW1X 9JF,2025-04-29T00:00:00,fhrs_5_en-gb,5,,FHRS,Kensington and Chelsea,-0.1578957,51.49866,0.0,0.0,0.0
15820,Temple Pier,Victoria Embankment,London,,Golden Spirit,Restaurant/Cafe/Canteen,1,0,NaN,1753160,24/00597/PORT,508,publicprotection@cityoflondon.gov.uk,City of London Corporation,http://www.cityoflondon.gov.uk/Corporation/hom...,False,,WC2R 2PN,2024-11-13T00:00:00,fhrs_5_en-gb,5,,FHRS,City of London Corporation,-0.1118936,51.5108151,0.0,0.0,0.0
58023,,745 Eastern Avenue,,Ilford,E N Market,Retailers - other,4613,0,NaN,607901,179975,526,themis.skouros@redbridge.gov.uk,Redbridge,http://www.redbridge.gov.uk/,False,,IG2 7RT,2026-03-09T00:00:00,fhrs_5_en-gb,5,,FHRS,Redbridge,0.094527,51.576122,5.0,5.0,5.0


In [5]:
# ==========================================
# Removing Duplicated Data
# ==========================================

print(f'Duplicated Rows = {df.duplicated().sum()}') # No. of duplicated data

df_drop = df.drop_duplicates() # Droping Rows found

Duplicated Rows = 1


In [6]:
# ==========================================
# Droping Unwanted Data
# ==========================================

print(df_drop.columns)

unwanted = columns=[
        'ChangesByServerID',
        'Distance',
        'LocalAuthorityBusinessID',
        'LocalAuthorityEmailAddress',
        'LocalAuthorityWebSite',
        'NewRatingPending',
        'Phone',
        'RatingKey',                        # List of Unwanted Cloumns
        'RightToReply',
        'SchemeType',
        'borough'
    ]

df_drop.drop(columns= unwanted, inplace= True) # Droping the listed Cloumns

print('Fix Applied')

print(df_drop.columns)

Index(['AddressLine1', 'AddressLine2', 'AddressLine3', 'AddressLine4',
       'BusinessName', 'BusinessType', 'BusinessTypeID', 'ChangesByServerID',
       'Distance', 'FHRSID', 'LocalAuthorityBusinessID', 'LocalAuthorityCode',
       'LocalAuthorityEmailAddress', 'LocalAuthorityName',
       'LocalAuthorityWebSite', 'NewRatingPending', 'Phone', 'PostCode',
       'RatingDate', 'RatingKey', 'RatingValue', 'RightToReply', 'SchemeType',
       'borough', 'geocode.longitude', 'geocode.latitude', 'scores.Hygiene',
       'scores.Structural', 'scores.ConfidenceInManagement'],
      dtype='object')
Fix Applied
Index(['AddressLine1', 'AddressLine2', 'AddressLine3', 'AddressLine4',
       'BusinessName', 'BusinessType', 'BusinessTypeID', 'FHRSID',
       'LocalAuthorityCode', 'LocalAuthorityName', 'PostCode', 'RatingDate',
       'RatingValue', 'geocode.longitude', 'geocode.latitude',
       'scores.Hygiene', 'scores.Structural', 'scores.ConfidenceInManagement'],
      dtype='object')


C:\Users\faiza\AppData\Local\Temp\ipykernel_16968\3203560544.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_drop.drop(columns= unwanted, inplace= True) # Droping the listed Cloumns


In [7]:
# ==========================================
# Header Cleanig
# ==========================================
print(df_drop.columns)

def to_snake_case(name):
    name = name.replace('.', '_')
    name = re.sub(r'(?<=[a-z0-9])(?=[A-Z])|(?<=[A-Z])(?=[A-Z][a-z])', '_', name)
    return name.lower()

df_drop.columns = [to_snake_case(c) for c in df_drop.columns]

print('Fix Applied')
print(df_drop.columns)

Index(['AddressLine1', 'AddressLine2', 'AddressLine3', 'AddressLine4',
       'BusinessName', 'BusinessType', 'BusinessTypeID', 'FHRSID',
       'LocalAuthorityCode', 'LocalAuthorityName', 'PostCode', 'RatingDate',
       'RatingValue', 'geocode.longitude', 'geocode.latitude',
       'scores.Hygiene', 'scores.Structural', 'scores.ConfidenceInManagement'],
      dtype='object')
Fix Applied
Index(['address_line1', 'address_line2', 'address_line3', 'address_line4',
       'business_name', 'business_type', 'business_type_id', 'fhrsid',
       'local_authority_code', 'local_authority_name', 'post_code',
       'rating_date', 'rating_value', 'geocode_longitude', 'geocode_latitude',
       'scores_hygiene', 'scores_structural',
       'scores_confidence_in_management'],
      dtype='object')


In [8]:
# ==========================================
#  Cleanig Empty Strings 
# ==========================================
df_drop = df_drop.replace('',pd.NA)

missing = df_drop.isnull().sum()
print(missing[missing>0])

address_line1                      22156
address_line2                      27868
address_line3                      35084
address_line4                      49878
post_code                           1087
geocode_longitude                  13622
geocode_latitude                   13622
scores_hygiene                     10688
scores_structural                  10688
scores_confidence_in_management    10688
dtype: int64


In [9]:
# ==========================================
# Handling Missing Values
# ==========================================

# Categorical Columns
cat_cols = ['address_line1','address_line2','address_line3','address_line4','post_code']


df_drop[cat_cols] = df_drop[cat_cols].fillna('Unknown')


# Numerical Columns
#num_cols = ['scores_hygiene','scores_structural','scores_confidence_in_management']

#df_drop[num_cols] = df_drop[num_cols].fillna()

In [10]:
# ==========================================
# Data type Correction
# ==========================================

df_drop['rating_date']= pd.to_datetime(df_drop['rating_date'], dayfirst=False, errors= 'coerce')  # Using datetime function

geocode = ['geocode_longitude','geocode_latitude']  # List of columns to be corrected

df_drop[geocode] = df_drop[geocode].apply(pd.to_numeric, errors='coerce')  # Converting to Numerical columns

df_drop[['rating_date', 'geocode_longitude','geocode_latitude']]

,rating_date,geocode_longitude,geocode_latitude
0,2024-07-13,0.142421,51.554493
1,2023-09-15,0.129392,51.558435
2,2023-09-02,0.110646,51.534718
3,2026-05-27,0.075250,51.541375
4,2024-07-13,0.128666,51.550930
...,...,...,...
79722,1901-01-01,-0.154266,51.513252
79723,1901-01-01,-0.132557,51.516397
79724,1901-01-01,-0.143139,51.516098
79725,1901-01-01,NaN,NaN


In [11]:
# ==========================================
# Making Overall_score column
# ==========================================
df_clean = df_drop.copy()

score = ['scores_hygiene','scores_structural','scores_confidence_in_management']

df_clean['overall_score'] = df_clean[score].mean(axis=1)



In [13]:
df_clean.sample(5)

,address_line1,address_line2,address_line3,address_line4,business_name,business_type,business_type_id,fhrsid,local_authority_code,local_authority_name,post_code,rating_date,rating_value,geocode_longitude,geocode_latitude,scores_hygiene,scores_structural,scores_confidence_in_management,overall_score
61365,Unknown,Unknown,Unknown,Unknown,Puddle Jumpers,Restaurant/Cafe/Canteen,1,1868121,527,Richmond-Upon-Thames,TW2,1901-01-01,AwaitingInspection,NaN,NaN,NaN,NaN,NaN,NaN
27185,7 Teal Street,Unknown,Greenwich,Unknown,Millennium Mini's Day Nursery,Caring Premises,5,695532,511,Greenwich,SE10 0ST,2025-05-20,5,0.014745,51.495334,0.0,0.0,0.0,0.000000
45947,245 Pavilion Road,LONDON,Unknown,Unknown,Natoora,Retailers - other,4613,922814,520,Kensington and Chelsea,SW1X 0BP,2026-01-29,5,-0.158553,51.493265,0.0,0.0,0.0,0.000000
12196,Unknown,4 Rosemont Road,Unknown,London,Osteria Ovada,Restaurant/Cafe/Canteen,1,1016130,506,Camden,NW3 6NE,2025-12-03,5,-0.182174,51.549272,5.0,5.0,5.0,5.000000
39753,Unknown,642 Uxbridge Road,Unknown,Hayes,Taste Of Jaffna,Takeaway/sandwich shop,7844,1687390,517,Hillingdon,UB4 0RY,2025-02-06,2,-0.412608,51.519694,10.0,15.0,10.0,11.666667


In [ ]:
%pip install sqlalchemy
%pip install psycopg2-binary


import os
from sqlalchemy import create_engine, URL


url = URL.create(
    drivername='postgresql+psycopg2',
    username='postgres',
    password=os.environ['PG_PASSWORD'],   
    host='localhost',
    port=5432,
    database='food_db',
)

engine = create_engine(url)


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [15]:
# 1. Write the table
df_clean.to_sql('fsa_rating', engine, if_exists='replace', index=False)

# 2. Verify
from sqlalchemy import inspect
insp = inspect(engine)
print(insp.get_table_names())

# 3. Check the data landed
pd.read_sql('SELECT COUNT(*) AS rows FROM fsa_rating', engine)

['fsa_rating']


,rows
0,79726
